In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [36]:
churn_df = pd.read_csv('/content/drive/MyDrive/ML Projects/churn_prediction/churn_dataset_changes.csv')

In [37]:
churn_df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,final_refund_amount,PhoneService,MultipleLines,InternetService,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn_binary,RelationCheck
0,7590-VHVEG,Female,0,Yes,No,1.0,0.000000,No,No phone service,DSL,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,True
1,5575-GNVDE,Male,0,No,No,34.0,0.000000,Yes,No,DSL,...,No,No,No,One year,No,Mailed check,56.95,1889.50,0,True
2,3668-QPYBK,Male,0,No,No,2.0,3713.570921,Yes,No,DSL,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,True
3,7795-CFOCW,Male,0,No,No,45.0,0.000000,No,No phone service,DSL,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0,True
4,9237-HQITU,Female,0,No,No,2.0,948.889474,Yes,No,Fiber optic,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,True


Data cleaning

In [38]:
drop_columns = {
    # "Churn_binary",
        "customerID",
        "final_refund_amount",
        "DeviceProtection",
        "InternetService",
        "StreamingMovies",
        "StreamingTV",
    
}

churn_df.drop(columns=drop_columns, inplace=True)
churn_df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,TechSupport,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn_binary,RelationCheck
0,Female,0,Yes,No,1.0,No,No phone service,No,Yes,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,True
1,Male,0,No,No,34.0,Yes,No,Yes,No,No,One year,No,Mailed check,56.95,1889.50,0,True
2,Male,0,No,No,2.0,Yes,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,True
3,Male,0,No,No,45.0,No,No phone service,Yes,No,Yes,One year,No,Bank transfer (automatic),42.30,1840.75,0,True
4,Female,0,No,No,2.0,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,True


In [39]:
churn_df['isMissing'] = churn_df.isnull().any(axis=1).astype(int)

# churn_df[churn_df['isMissing'] == True]
# churn_df.dropna(inplace=True)
churn_df.head(100)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,TechSupport,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn_binary,RelationCheck,isMissing
0,Female,0,Yes,No,1.0,No,No phone service,No,Yes,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,True,0
1,Male,0,No,No,34.0,Yes,No,Yes,No,No,One year,No,Mailed check,56.95,1889.50,0,True,0
2,Male,0,No,No,2.0,Yes,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,True,0
3,Male,0,No,No,45.0,No,No phone service,Yes,No,Yes,One year,No,Bank transfer (automatic),42.30,1840.75,0,True,0
4,Female,0,No,No,2.0,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Female,0,No,No,29.0,Yes,Yes,Yes,No,No,Month-to-month,Yes,Electronic check,70.50,1401.15,1,False,0
96,Male,0,Yes,Yes,29.0,Yes,Yes,Yes,Yes,Yes,One year,Yes,Credit card (automatic),70.50,1401.15,0,False,0
97,Male,0,No,No,5.0,Yes,No,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,21.05,113.85,1,True,0
98,Male,0,No,No,52.0,Yes,No,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),21.00,1107.20,0,True,0


In [40]:
median_monthly_charges = churn_df['MonthlyCharges'].median()
churn_df['MonthlyCharges'].fillna(median_monthly_charges, inplace=True)

median_total_charges = churn_df['TotalCharges'].median()
churn_df['TotalCharges'].fillna(median_total_charges, inplace=True)

median_tenure = churn_df['tenure'].median()
churn_df['tenure'].fillna(median_tenure, inplace=True)
churn_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   float64
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   OnlineSecurity    7043 non-null   object 
 8   OnlineBackup      7043 non-null   object 
 9   TechSupport       7043 non-null   object 
 10  Contract          7043 non-null   object 
 11  PaperlessBilling  7043 non-null   object 
 12  PaymentMethod     7043 non-null   object 
 13  MonthlyCharges    7043 non-null   float64
 14  TotalCharges      7043 non-null   float64
 15  Churn_binary      7043 non-null   int64  
 16  RelationCheck     7043 non-null   bool   


/tmp/ipython-input-1431913328.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  churn_df['MonthlyCharges'].fillna(median_monthly_charges, inplace=True)
/tmp/ipython-input-1431913328.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value,

In [41]:
# churn_df.